In [1]:
# pip install opensmile

In [2]:
#pip install praat-parselmouth

In [3]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
from tqdm import tqdm

from scipy.stats.mstats import zscore
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

import opensmile

import os
import data_loader_HUMV
import audio_preprocessing
from feature_extraction_pipeline import *


In [4]:
import warnings
warnings.filterwarnings("ignore")

In [5]:
# # Define the root directory where the audio files are stored
# root_directory = r'C:\Users\Marcos\Documents\IDIVAL\Proyectos\Audio_Parkinson\Audio_HUMV_enhanced'
# start_with=  'pataka' #'vocal'
# exact_name =    None #'pataka.wav'

# # Load the audio data using the data_loader function
# df = data_loader_HUMV.load_audio_data(root_directory, start_with, exact_name)
# df.head()

# # Patients to drop
# patients_to_drop = []

# # Drop rows where 'Patient' is in the list of patients_to_drop
# df_filtered = df[~df['Patient'].isin(patients_to_drop)]

# # Show the first few rows of the loaded DataFrame
# df_filtered.head()

# # Step 1: Filter the rows where the Label is either 0 or 2
# df_HC_vs_PD = df_filtered[df_filtered['Label'].isin([0, 2])]

# # Step 2: Replace the Label value 2 with 1
# df_HC_vs_PD['Label'] = df_HC_vs_PD['Label'].replace(2, 1)

# df_HC_vs_PD.shape

# df_HC_vs_PD['Label'].value_counts()

# preprocessed_chunks_data_np, labels_chunks_np, ids_np = audio_preprocessing.execute_preprocess_and_split(df_HC_vs_PD, start_time=0, chunk_duration=5, max_duration = None, target_sr=16000, remove_silence= True, file_path_column='File_Path')

# # len preprocessed_chunks_data_np[0] = target sr (16000) x 5 s = 80000 
# len(preprocessed_chunks_data_np[0])

# # Get unique values and their counts
# unique_labels, counts = np.unique(labels_chunks_np, return_counts=True)

# # Print unique values and their counts
# for label, count in zip(unique_labels, counts):
#     print(f"Label {label}: {count} occurrences")


# len(ids_np), len(labels_chunks_np)

# OpenSmile

In [6]:
# smile = opensmile.Smile(
#     feature_set=opensmile.FeatureSet.ComParE_2016,
#     feature_level=opensmile.FeatureLevel.Functionals,
# )

# # Initialize a list to store features for each chunk
# all_chunk_features = []

# # Loop through each chunk and process the signal extracting features with OpenSmile
# for i, chunk in tqdm(enumerate(preprocessed_chunks_data_np), total=len(preprocessed_chunks_data_np), desc="Processing chunks"):
#     # Extract features for the current chunk
#     features = smile.process_signal(chunk, sampling_rate=16000)
    
#     # Convert the features (if not already a DataFrame) and append to the list
#     if not isinstance(features, pd.DataFrame):
#         features = pd.DataFrame(features)
    
#     all_chunk_features.append(features)

# # Combine all chunk features into a single DataFrame
# training_features_df = pd.concat(all_chunk_features, ignore_index=True)

# training_features_df.shape

# # Check for NaN values in the entire DataFrame
# nan_summary = training_features_df.isna().sum()

# if len(nan_summary[nan_summary > 0]) > 0:
#     # Print the columns that have missing values along with the count
#     print("Columns with missing values and the count of NaNs in each column:")
#     print(nan_summary[nan_summary > 0])
# else:
#     print("There are no columns with missing values")

# Praat

In [7]:
# import parselmouth

# # Initialize a list to store features for each chunk
# all_chunk_features = []

# # Loop through each chunk and process the signal extracting features with Praat
# for i, chunk in tqdm(enumerate(preprocessed_chunks_data_np), total=len(preprocessed_chunks_data_np), desc="Processing chunks with Praat"):
#     try:
#         # Create Sound object
#         sound = parselmouth.Sound(chunk, sampling_rate=16000)

#         # Pitch
#         pitch = sound.to_pitch()
#         pitch_values = pitch.selected_array['frequency']
#         pitch_values = pitch_values[pitch_values != 0]  # Remove unvoiced
#         pitch_mean = np.mean(pitch_values) if len(pitch_values) > 0 else 0

#         # Formants
#         formants = sound.to_formant_burg()
#         f1_values = []
#         for t in formants.t_grid():
#             f1 = formants.get_value_at_time(1, t)
#             if f1 != 0:
#                 f1_values.append(f1)
#         f1_mean = np.mean(f1_values) if f1_values else 0

#         # Intensity
#         intensity = sound.to_intensity()
#         intensity_values = intensity.values[0]
#         intensity_mean = np.mean(intensity_values)

#         # Jitter and Shimmer
#         point_process = parselmouth.praat.call(sound, "To PointProcess (periodic, cc)", 75, 600)
#         jitter = parselmouth.praat.call(point_process, "Get jitter (local)", 0, 0, 0.0001, 0.02, 1.3)

#         # Harmonics-to-Noise Ratio
#         hnr = sound.to_harmonicity_cc()
#         hnr_values = hnr.values[0]
#         hnr_mean = np.mean(hnr_values)

#         # Collect features
#         features = {
#             'pitch_mean': pitch_mean,
#             'f1_mean': f1_mean,
#             'intensity_mean': intensity_mean,
#             'jitter': jitter,
#             'hnr_mean': hnr_mean
#         }
#         all_chunk_features.append(features)

#     except Exception as e:
#         print(f"Error processing chunk {i}: {e}")
#         all_chunk_features.append({k: np.nan for k in ['pitch_mean', 'f1_mean', 'intensity_mean', 'jitter', 'hnr_mean']})

# # Create DataFrame
# features_training_df = pd.DataFrame(all_chunk_features)

# features_training_df.shape

# # Check for NaN values
# nan_summary = features_training_df.isna().sum()
# if len(nan_summary[nan_summary > 0]) > 0:
#     print("Columns with missing values and the count of NaNs in each column:")
#     print(nan_summary[nan_summary > 0])
# else:
#     print("There are no columns with missing values")

# Simplified extraction

In [8]:
from feature_extraction_pipeline import load_and_preprocess_data, extract_features_opensmile, extract_features_praat

# Define the root directory where the audio files are stored
root_directory = r'C:\Users\Marcos\Documents\IDIVAL\Proyectos\Audio_Parkinson\Audio_HUMV_enhanced'
start_with=  'pataka' #'vocal'
exact_name =    None #'pataka.wav'

# For 'pataka' exercise
chunks, labels, ids, df = load_and_preprocess_data(root_directory, start_with='pataka')

# OpenSMILE features
# features_df_os = extract_features_opensmile(chunks, labels, ids, save_df=True, output_path='features_pataka_opensmile.csv')

# Praat features
features_df_pr = extract_features_praat(chunks, labels, ids, save_df=True, output_path='features_pataka_praat.csv')

# # For 'vocal' exercise
# chunks, labels, ids, df = load_and_preprocess_data(root_directory, start_with='vocal')

# # OpenSMILE features
# features_df_os = extract_features_opensmile(chunks, labels, ids, save_df=True, output_path='features_vocal_opensmile.csv')

# # Praat features
# features_df_pr = extract_features_praat(chunks, labels, ids, save_df=True, output_path='features_vocal_praat.csv')

Loaded 117 audio files.
Label distribution:
Label
0    50
1    34
2    33
Name: count, dtype: int64
DataFrame shape after filtering: (83, 4)
Label distribution:
Label
0    50
1    33
Name: count, dtype: int64
Total chunks generated: 98
Number of chunks: 98
Chunk length (samples): 80000
Expected chunk length: 80000
Label 0: 62 occurrences
Label 1: 36 occurrences


Processing chunks with Praat: 100%|██████████| 98/98 [00:04<00:00, 23.08it/s]

Praat Features DataFrame shape: (98, 14)
There are no columns with missing values
Features saved to features_pataka_praat.csv
